***Script used to download fruit fly hemibrain data and to filter fly connectivity to match sweat bee ROI***

Also used for:

- Fruit fly anterior visual pathway cell quantities (Fig. 1)

In [2]:
import pandas as pd
import numpy as np
import json

import neuprint as neu
from neuprint import fetch_synapses, fetch_synapse_connections, NeuronCriteria as NC, SynapseCriteria as SC
from neuprint.client import setup_debug_logging
import navis
import navis.interfaces.neuprint as n_neu

from scipy.spatial import ConvexHull, convex_hull_plot_2d
import trimesh

In [13]:
# replace 'token' below with your token
# See https://connectome-neuprint.github.io/neuprint-python/docs/quickstart.html

path = 'D:/flywire_backup/cave/neuprint.json' # replace with your path

with open(path, 'r') as f:
    token_file = json.load(f)
token = token_file['token']

client = neu.Client(
    "https://neuprint.janelia.org/",
    token=token,  
    dataset="hemibrain:v1.2.1"
)

In [62]:
# Show the ROI hierarchy, with primary ROIs marked with '*'
print(neu.fetch_roi_hierarchy(include_subprimary=True, mark_primary=True, format='text'))

hemibrain
 +-- AL(L)*
 |   +-- AL-D(L)
 |   +-- AL-DA2(L)
 |   +-- AL-DA3(L)
 |   +-- AL-DA4m(L)
 |   +-- AL-DC1(L)
 |   +-- AL-DC2(L)
 |   +-- AL-DC4(L)
 |   +-- AL-DL4(L)
 |   +-- AL-DL5(L)
 |   +-- AL-DM1(L)
 |   +-- AL-DM2(L)
 |   +-- AL-DM3(L)
 |   +-- AL-DM4(L)
 |   +-- AL-DM5(L)
 |   +-- AL-DM6(L)
 |   +-- AL-DP1m(L)
 |   +-- AL-VA6(L)
 |   +-- AL-VM7d(L)
 |   +-- AL-VM7v(L)
 +-- AL(R)*
 |   +-- AL-D(R)
 |   +-- AL-DA1(R)
 |   +-- AL-DA2(R)
 |   +-- AL-DA3(R)
 |   +-- AL-DA4l(R)
 |   +-- AL-DA4m(R)
 |   +-- AL-DC1(R)
 |   +-- AL-DC2(R)
 |   +-- AL-DC3(R)
 |   +-- AL-DC4(R)
 |   +-- AL-DL1(R)
 |   +-- AL-DL2d(R)
 |   +-- AL-DL2v(R)
 |   +-- AL-DL3(R)
 |   +-- AL-DL4(R)
 |   +-- AL-DL5(R)
 |   +-- AL-DM1(R)
 |   +-- AL-DM2(R)
 |   +-- AL-DM3(R)
 |   +-- AL-DM4(R)
 |   +-- AL-DM5(R)
 |   +-- AL-DM6(R)
 |   +-- AL-DP1l(R)
 |   +-- AL-DP1m(R)
 |   +-- AL-V(R)
 |   +-- AL-VA1d(R)
 |   +-- AL-VA1v(R)
 |   +-- AL-VA2(R)
 |   +-- AL-VA3(R)
 |   +-- AL-VA4(R)
 |   +-- AL-VA5(R)
 |   +-- A

In [ ]:
source = NC(type='EPG.*|PEG.*|PEN_a.*|PEN_b.*|Delta7.*|ER.*|GLNO.*|LNO.*')
source_info, source_conn = neu.fetch_neurons(source)

target =  NC(type='EPG.*|PEG.*|PEN_a.*|PEN_b.*|Delta7.*|ER.*|GLNO.*|LNO.*')
target_info, target_conn = neu.fetch_neurons(target)

source_info # check 

In [26]:
# for fly synapse table
# Merge 'type' and 'instance' from source_info into source_conn based on 'bodyId'

source_conn = source_conn.merge(
    source_info[['bodyId', 'type', 'instance']],
    on='bodyId',
    how='left'  # Use 'inner' if you only want rows that match
)
source_conn

,bodyId,roi,pre,post,downstream,upstream,mito,type,instance
0,387023620,ATL(L),0,5,0,5,0,PEN_b(PEN2),PEN_b(PB06b)_L4
1,387023620,CX,341,1703,1653,1703,142,PEN_b(PEN2),PEN_b(PB06b)_L4
2,387023620,EB,233,559,1134,559,53,PEN_b(PEN2),PEN_b(PB06b)_L4
3,387023620,EBr1,0,5,0,5,0,PEN_b(PEN2),PEN_b(PB06b)_L4
4,387023620,EBr2r4,7,22,36,22,1,PEN_b(PEN2),PEN_b(PB06b)_L4
...,...,...,...,...,...,...,...,...,...
6001,5813080979,NotPrimary,0,5,0,5,71,PEN_a(PEN1),PEN_a(PB06a)_L5
6002,5813080979,PB,0,451,0,451,78,PEN_a(PEN1),PEN_a(PB06a)_L5
6003,5813080979,PB(L4),0,37,0,37,2,PEN_a(PEN1),PEN_a(PB06a)_L5
6004,5813080979,PB(L5),0,408,0,408,76,PEN_a(PEN1),PEN_a(PB06a)_L5


In [45]:
# for fly conntable
source_list = source_info['bodyId'].tolist()
target_list = target_info['bodyId'].tolist()
neuron_df, conn_df = neu.fetch_adjacencies(sources=source_list, targets=target_list)

conn_df = neu.merge_neuron_properties(neuron_df, conn_df, ['type', 'instance'])

unique_count_source = source_info['bodyId'].nunique()
unique_count_target = target_info['bodyId'].nunique()
unique_count_fetched = neuron_df['bodyId'].nunique()

print(f'{unique_count_source} unique source neurons, {unique_count_source} unique target, {unique_count_fetched} neurons fetched total')

  0%|          | 0/3 [00:00<?, ?it/s]

422 unique source neurons, 422 unique target, 422 neurons fetched total


In [28]:
conn_df

,bodyId_pre,bodyId_post,roi,weight,type_pre,instance_pre,type_post,instance_post
0,387023620,387364605,EB,66,PEN_b(PEN2),PEN_b(PB06b)_L4,EPG,EPG(PB08)_L3
1,387023620,449438847,EB,52,PEN_b(PEN2),PEN_b(PB06b)_L4,EPG,EPG(PB08)_L3
2,387023620,508793049,NO,6,PEN_b(PEN2),PEN_b(PB06b)_L4,PEN_a(PEN1),PEN_a(PB06a)_L7
3,387023620,539462336,NO,30,PEN_b(PEN2),PEN_b(PB06b)_L4,PEN_b(PEN2),PEN_b(PB06b)_L4
4,387023620,539462336,EB,8,PEN_b(PEN2),PEN_b(PB06b)_L4,PEN_b(PEN2),PEN_b(PB06b)_L4
...,...,...,...,...,...,...,...,...
46043,5813080979,5813047793,EB,2,PEN_a(PEN1),PEN_a(PB06a)_L5,ER4m,ER4m(ring)_L
46044,5813080979,5813049948,EB,2,PEN_a(PEN1),PEN_a(PB06a)_L5,ER4m,ER4m(ring)_L
46045,5813080979,5813053866,EB,6,PEN_a(PEN1),PEN_a(PB06a)_L5,ER4m,ER4m(ring)_R
46046,5813080979,5813059835,EB,4,PEN_a(PEN1),PEN_a(PB06a)_L5,ER1_a,ER1_a(ring)_L


In [46]:
# goal obtain total number of postsynaptic synapses within a ROI for each neuron
# this will be used to calculate the relative weights
# add total postsynaptic column total_post_syn
neuron_info = pd.concat([source_info, target_info])

flat_rows = []
for entry in neuron_info.roiInfo:
    flat_row = {}
    for roi, counts in entry.items():
        for k, v in counts.items():
            flat_row[f"{roi}_{k}"] = v
    flat_rows.append(flat_row)

flat_df = pd.DataFrame(flat_rows).reset_index(drop=True)
neuron_info = neuron_info.reset_index(drop=True)

neuron_info['PB_post'] = flat_df['PB_post']
neuron_info['EB_post'] = flat_df['EB_post']
neuron_info['NO_post'] = flat_df['NO_post']

neuron_info_unique = neuron_info[['bodyId','PB_post','EB_post','NO_post']].drop_duplicates('bodyId') # drop duplicates

example_neuron = neuron_info[neuron_info['bodyId'] == 508793049]
print(example_neuron['NO_post'])


5      505.0
427    505.0
Name: NO_post, dtype: float64


In [47]:
# perform a left join using bodyId_post <-> bodyId
conn_df = conn_df.merge(
    neuron_info_unique[['bodyId', 'PB_post', 'EB_post', 'NO_post']],   # roi_post == total # of syn in that ROI
    left_on='bodyId_post', 
    right_on='bodyId', 
    how='left'
)

# drop the redundant 'bodyId' column added by the merge
conn_df = conn_df.drop(columns=['bodyId'])

cols = ['PB_post', 'EB_post', 'NO_post']
conn_df[cols] = conn_df[cols].fillna(0).astype(int)

conn_df

,bodyId_pre,bodyId_post,roi,weight,type_pre,instance_pre,type_post,instance_post,PB_post,EB_post,NO_post
0,387023620,387364605,EB,66,PEN_b(PEN2),PEN_b(PB06b)_L4,EPG,EPG(PB08)_L3,265,3892,0
1,387023620,449438847,EB,52,PEN_b(PEN2),PEN_b(PB06b)_L4,EPG,EPG(PB08)_L3,331,4070,0
2,387023620,508793049,NO,6,PEN_b(PEN2),PEN_b(PB06b)_L4,PEN_a(PEN1),PEN_a(PB06a)_L7,274,814,505
3,387023620,539462336,NO,30,PEN_b(PEN2),PEN_b(PB06b)_L4,PEN_b(PEN2),PEN_b(PB06b)_L4,698,520,388
4,387023620,539462336,EB,8,PEN_b(PEN2),PEN_b(PB06b)_L4,PEN_b(PEN2),PEN_b(PB06b)_L4,698,520,388
...,...,...,...,...,...,...,...,...,...,...,...
46043,5813080979,5813047793,EB,2,PEN_a(PEN1),PEN_a(PB06a)_L5,ER4m,ER4m(ring)_L,0,1522,0
46044,5813080979,5813049948,EB,2,PEN_a(PEN1),PEN_a(PB06a)_L5,ER4m,ER4m(ring)_L,0,1562,0
46045,5813080979,5813053866,EB,6,PEN_a(PEN1),PEN_a(PB06a)_L5,ER4m,ER4m(ring)_R,0,1685,0
46046,5813080979,5813059835,EB,4,PEN_a(PEN1),PEN_a(PB06a)_L5,ER1_a,ER1_a(ring)_L,0,449,0


In [48]:
fly_count = conn_df.copy()

fly_count = fly_count.rename(columns={'weight':'count'})
# (don't do) update PEN_a to PEN1 and PEN_b to PEN2
# fly_count['instance_pre'] = fly_count['instance_pre'].str.replace(r'^PEN_a', 'PEN1', regex=True)
# fly_count['instance_pre'] = fly_count['instance_pre'].str.replace(r'^PEN_b', 'PEN2', regex=True)
# fly_count['instance_post'] = fly_count['instance_post'].str.replace(r'^PEN_a', 'PEN1', regex=True)
# fly_count['instance_post'] = fly_count['instance_post'].str.replace(r'^PEN_b', 'PEN2', regex=True)

# fly_count['type_pre'] = fly_count['type_pre'].str.replace(r'^PEN_a', 'PEN1', regex=True)
# fly_count['type_pre'] = fly_count['type_pre'].str.replace(r'^PEN_b', 'PEN2', regex=True)
# fly_count['type_post'] = fly_count['type_post'].str.replace(r'^PEN_a', 'PEN1', regex=True)
# fly_count['type_post'] = fly_count['type_post'].str.replace(r'^PEN_b', 'PEN2', regex=True)

####################################
####################################

# add column for relative weights: take synapses from neuron a to neuron b in region X and divided this 
# number by the total number of inputs that neuron b received in X

fly_count['bodyId_pre'] = fly_count['bodyId_pre'].astype(float).astype(int)
fly_count['bodyId_post'] = fly_count['bodyId_post'].astype(float).astype(int)

def compute_relative_weight(row, roi_name):
    if row['roi'] == roi_name:
        denom = row.get(f'{roi_name}_post', np.nan)
        if denom == 0 or pd.isna(denom):
            return 0.0
        return row['count'] / denom
    else:
        return 0.0

fly_count['relative_weight_PB'] = fly_count.apply(lambda row: compute_relative_weight(row, 'PB'), axis=1)
fly_count['relative_weight_EB'] = fly_count.apply(lambda row: compute_relative_weight(row, 'EB'), axis=1)
fly_count['relative_weight_NO'] = fly_count.apply(lambda row: compute_relative_weight(row, 'NO'), axis=1)


# Add threshold (greater than 5 synapses and 0.01 relative weight)
fly_count = fly_count[fly_count['count']>5]
# fly_count = fly_count[fly_count['relative_weight']>0.01]

# ####################################
# ####################################
# Regex patterns
pattern_standard = r'^([A-Z]{3}[a-zA-Z]?(?:\d+)?(?:_[a-z])?).*(R\d|L\d)'
pattern_er = r'^(ER\d+[a-z]?)(?:_([a-z]))?\(ring\)_([LR])$'
pattern_d7 = r'(?i)(Delta7).*?_(L\d(?:[LR]\d)+)'
pattern_lno = r'(LNO3|LNO2|LNO1|LNOa|GLNO)'

# Extract for standard neurons
extracted_pre_standard = fly_count['instance_pre'].str.extract(pattern_standard)
extracted_post_standard = fly_count['instance_post'].str.extract(pattern_standard)

# Extract for delta7 neurons
extracted_pre_d7 = fly_count['instance_pre'].str.extract(pattern_d7)
extracted_post_d7 = fly_count['instance_post'].str.extract(pattern_d7)

# Extract for lno neurons
extracted_lno_pre = fly_count['type_pre'].str.extract(pattern_lno)
extracted_lno_post = fly_count['type_post'].str.extract(pattern_lno)

# Extract for ER neurons
extracted_pre_er = fly_count['instance_pre'].str.extract(pattern_er)
extracted_post_er = fly_count['instance_post'].str.extract(pattern_er)


# precompute the fully constructed strings for each pattern
d7_type_pre = extracted_pre_d7[0] + '_' + extracted_pre_d7[1]
d7_type_post = extracted_post_d7[0] + '_' + extracted_post_d7[1]

er_type_pre = extracted_pre_er[0] + extracted_pre_er[1].fillna('') + '_' + extracted_pre_er[2]
er_type_post = extracted_post_er[0] + extracted_post_er[1].fillna('') + '_' + extracted_post_er[2]

std_type_pre = extracted_pre_standard[0] + '_' + extracted_pre_standard[1]
std_type_post = extracted_post_standard[0] + '_' + extracted_post_standard[1]


# assign type (e.g., only the class, not location info)
fly_count['type_pre'] = extracted_pre_er[0].combine_first(extracted_pre_standard[0]).combine_first(extracted_pre_d7[0]).combine_first(extracted_lno_pre[0])
fly_count['type_post'] = extracted_post_er[0].combine_first(extracted_post_standard[0]).combine_first(extracted_post_d7[0]).combine_first(extracted_lno_post[0])

fly_count['type_pre_col'] = er_type_pre.combine_first(std_type_pre).combine_first(d7_type_pre).combine_first(extracted_lno_pre[0])
fly_count['type_post_col'] = er_type_post.combine_first(std_type_post).combine_first(d7_type_post).combine_first(extracted_lno_post[0])

# # final naming
fly_count['pre_name'] = fly_count['type_pre_col'] + '_' + fly_count['bodyId_pre'].astype(str)
fly_count['post_name'] = fly_count['type_post_col'] + '_' + fly_count['bodyId_post'].astype(str)

fly_count

,bodyId_pre,bodyId_post,roi,count,type_pre,instance_pre,type_post,instance_post,PB_post,EB_post,NO_post,relative_weight_PB,relative_weight_EB,relative_weight_NO,type_pre_col,type_post_col,pre_name,post_name
0,387023620,387364605,EB,66,PEN_b,PEN_b(PB06b)_L4,EPG,EPG(PB08)_L3,265,3892,0,0.0,0.016958,0.000000,PEN_b_L4,EPG_L3,PEN_b_L4_387023620,EPG_L3_387364605
1,387023620,449438847,EB,52,PEN_b,PEN_b(PB06b)_L4,EPG,EPG(PB08)_L3,331,4070,0,0.0,0.012776,0.000000,PEN_b_L4,EPG_L3,PEN_b_L4_387023620,EPG_L3_449438847
2,387023620,508793049,NO,6,PEN_b,PEN_b(PB06b)_L4,PEN_a,PEN_a(PB06a)_L7,274,814,505,0.0,0.000000,0.011881,PEN_b_L4,PEN_a_L7,PEN_b_L4_387023620,PEN_a_L7_508793049
3,387023620,539462336,NO,30,PEN_b,PEN_b(PB06b)_L4,PEN_b,PEN_b(PB06b)_L4,698,520,388,0.0,0.000000,0.077320,PEN_b_L4,PEN_b_L4,PEN_b_L4_387023620,PEN_b_L4_539462336
4,387023620,539462336,EB,8,PEN_b,PEN_b(PB06b)_L4,PEN_b,PEN_b(PB06b)_L4,698,520,388,0.0,0.015385,0.000000,PEN_b_L4,PEN_b_L4,PEN_b_L4_387023620,PEN_b_L4_539462336
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46039,5813080979,5813022281,EB,60,PEN_a,PEN_a(PB06a)_L5,EPG,EPG(PB08)_L4,271,2836,0,0.0,0.021157,0.000000,PEN_a_L5,EPG_L4,PEN_a_L5_5813080979,EPG_L4_5813022281
46040,5813080979,5813027103,EB,20,PEN_a,PEN_a(PB06a)_L5,EPG,EPG(PB08)_R5,273,3286,0,0.0,0.006086,0.000000,PEN_a_L5,EPG_R5,PEN_a_L5_5813080979,EPG_R5_5813027103
46041,5813080979,5813047157,NO,7,PEN_a,PEN_a(PB06a)_L5,PEN_b,PEN_b(PB06b)_L2,753,679,524,0.0,0.000000,0.013359,PEN_a_L5,PEN_b_L2,PEN_a_L5_5813080979,PEN_b_L2_5813047157
46045,5813080979,5813053866,EB,6,PEN_a,PEN_a(PB06a)_L5,ER4m,ER4m(ring)_R,0,1685,0,0.0,0.003561,0.000000,PEN_a_L5,ER4m_R,PEN_a_L5_5813080979,ER4m_R_5813053866


In [21]:
# repeat for syntable
# Regex patterns
pattern_standard = r'^([A-Z]{3}[a-zA-Z]?(?:\d+)?(?:_[a-z])?).*(R\d|L\d)'
pattern_er = r'^(ER\d+[a-z]?)(?:_([a-z]))?\(ring\)_([LR])$'
pattern_d7 = r'(?i)(Delta7).*?_(L\d(?:[LR]\d)+)'
pattern_lno = r'(LNO3|LNO2|LNO1|LNOa|GLNO)'

fly_syn = source_conn.copy()

# Extract for standard neurons
extracted_standard = fly_syn['instance'].str.extract(pattern_standard)

# Extract for er neurons
extracted_er = fly_syn['instance'].str.extract(pattern_er)

# Extract for d7 neurons
extracted_d7 = fly_syn['instance'].str.extract(pattern_d7)

# Extract for lno neurons
extracted_lno = fly_syn['type'].str.extract(pattern_lno)

# precompute the fully constructed strings for each pattern
d7_type = extracted_d7[0] + '_' + extracted_d7[1]
er_type = extracted_er[0] + extracted_er[1].fillna('') + '_' + extracted_er[2]
std_type = extracted_standard[0] + '_' + extracted_standard[1]

# assign type (e.g., only the class, not location info)
fly_syn['type'] = extracted_er[0].combine_first(extracted_standard[0]).combine_first(extracted_d7[0]).combine_first(extracted_lno[0])

fly_syn['type_col'] = er_type.combine_first(std_type).combine_first(d7_type).combine_first(extracted_lno[0])

# # final naming
fly_syn['name'] = fly_syn['type_col'] + '_' + fly_syn['bodyId'].astype(str)

fly_syn

,bodyId,roi,pre,post,downstream,upstream,mito,type,instance,type_col,name
0,387023620,ATL(L),0,5,0,5,0,PEN_b,PEN_b(PB06b)_L4,PEN_b_L4,PEN_b_L4_387023620
1,387023620,CX,341,1703,1653,1703,142,PEN_b,PEN_b(PB06b)_L4,PEN_b_L4,PEN_b_L4_387023620
2,387023620,EB,233,559,1134,559,53,PEN_b,PEN_b(PB06b)_L4,PEN_b_L4,PEN_b_L4_387023620
3,387023620,EBr1,0,5,0,5,0,PEN_b,PEN_b(PB06b)_L4,PEN_b_L4,PEN_b_L4_387023620
4,387023620,EBr2r4,7,22,36,22,1,PEN_b,PEN_b(PB06b)_L4,PEN_b_L4,PEN_b_L4_387023620
...,...,...,...,...,...,...,...,...,...,...,...
6001,5813080979,NotPrimary,0,5,0,5,71,PEN_a,PEN_a(PB06a)_L5,PEN_a_L5,PEN_a_L5_5813080979
6002,5813080979,PB,0,451,0,451,78,PEN_a,PEN_a(PB06a)_L5,PEN_a_L5,PEN_a_L5_5813080979
6003,5813080979,PB(L4),0,37,0,37,2,PEN_a,PEN_a(PB06a)_L5,PEN_a_L5,PEN_a_L5_5813080979
6004,5813080979,PB(L5),0,408,0,408,76,PEN_a,PEN_a(PB06a)_L5,PEN_a_L5,PEN_a_L5_5813080979


In [ ]:
# check for and remove duplicates if necessary
# fly_all = fly_all[~fly_all.duplicated(subset=['bodyId_pre', 'bodyId_post', 'roi', 'count'])]


***IMPORTANT: for comparing synapses between sweat bee and fly, use 'downstream' and 'upstream', not 'pre' and 'post'***

In [49]:
# export
#fly_syn.to_csv('./table_csv_files/fly_hemibrain_all_hd_cells_syntable.csv', index=False) # save for count statistics
fly_count.to_csv('./table_csv_files/fly_hemibrain_all_hd_cells_conntable.csv', index=False)

### filter fly syntable and connectivity table to match ROI included in sweat bee analysis

Fly hemibrain columnar cells have annotations for PB columns, but not for EB columns. Below, I use the branches of EPGs within the EB to extract a subvolume that contains EBc1 and EBc2.

In [54]:
# to obtain synapse counts

In [126]:
# import synapse dataframe
fly_syn_df = pd.read_csv('../syntables/table_csv_files/fly_hemibrain_all_hd_cells_syntable_simplified.csv')

In [127]:
fly_syn_df

,bodyId,roi,pre,post,downstream,upstream,mito,type,instance,type_col,name
0,387023620,ATL(L),0,5,0,5,0,PEN_b,PEN_b(PB06b)_L4,PEN_b_L4,PEN_b_L4_387023620
1,387023620,CX,341,1703,1653,1703,142,PEN_b,PEN_b(PB06b)_L4,PEN_b_L4,PEN_b_L4_387023620
2,387023620,EB,233,559,1134,559,53,PEN_b,PEN_b(PB06b)_L4,PEN_b_L4,PEN_b_L4_387023620
3,387023620,EBr1,0,5,0,5,0,PEN_b,PEN_b(PB06b)_L4,PEN_b_L4,PEN_b_L4_387023620
4,387023620,EBr2r4,7,22,36,22,1,PEN_b,PEN_b(PB06b)_L4,PEN_b_L4,PEN_b_L4_387023620
...,...,...,...,...,...,...,...,...,...,...,...
6001,5813080979,NotPrimary,0,5,0,5,71,PEN_a,PEN_a(PB06a)_L5,PEN_a_L5,PEN_a_L5_5813080979
6002,5813080979,PB,0,451,0,451,78,PEN_a,PEN_a(PB06a)_L5,PEN_a_L5,PEN_a_L5_5813080979
6003,5813080979,PB(L4),0,37,0,37,2,PEN_a,PEN_a(PB06a)_L5,PEN_a_L5,PEN_a_L5_5813080979
6004,5813080979,PB(L5),0,408,0,408,76,PEN_a,PEN_a(PB06a)_L5,PEN_a_L5,PEN_a_L5_5813080979


In [6]:
EB = neu.fetch_roi("EB")
PB = neu.fetch_roi("PB")

In [40]:
EB = neu.fetch_roi("EB")
PB = neu.fetch_roi("PB")


fly_EB_filt = fly_syn_df[fly_syn_df['roi'].str.contains(r'^EB$')]
epg = r'EPGt_R9|EPG_R9|EPG_R8|EPG_L1|EPG_L2'
peg = r'PEG_R9|PEG_R8|PEG_L1|PEG_L2'
# pen_a = r'PEN_a_R9|PEN_a_R8|PEN_a_L2|PEN_a_L3'
# pen_b = r'PEN_b_R9|PEN_b_R8|PEN_b_L2|PEN_b_L3'

    
fly_EB_filt = fly_EB_filt[(fly_EB_filt['type_col'].str.contains(epg, case=False) | #filter type_pre_colaptic partner
                    fly_EB_filt['type_col'].str.contains(peg, case=False)
                    )]

# fly_EB_filt = fly_syn_df[fly_syn_df['roi'].str.contains(r'^EB$')]
# epg = r'EPGt_R9|EPG_R9|EPG_R8|EPG_L1|EPG_L2'
# peg = r'PEG_R9|PEG_R8|PEG_L1|PEG_L2'
# pen_a = r'PEN_a_R9|PEN_a_R8|PEN_a_L2|PEN_a_L3'
# pen_b = r'PEN_b_R9|PEN_b_R8|PEN_b_L2|PEN_b_L3'

    
fly_EB_filt = fly_EB_filt[(fly_EB_filt['type_col'].str.contains(epg, case=False) | #filter type_pre_colaptic partner
                    fly_EB_filt['type_col'].str.contains(peg, case=False)
                    )]


In [41]:
eb_lateral = fly_EB_filt['bodyId'].tolist()

eb_sklist = neu.fetch_skeletons( # download full skeletons
    neu.SegmentCriteria(bodyId=eb_lateral, regex=True), with_synapses=True
)

eb_sklist

Fetching:   0%|          | 0/14 [00:00<?, ?it/s]

,type,name,id,n_nodes,n_connectors,n_branches,n_leafs,cable_length,soma,units
0,navis.TreeNeuron,EPG(PB08)_L1,572870540,14361,5572,1616,1658,541049.87500,14150,8 nanometer
1,navis.TreeNeuron,EPG(PB08)_L2,697001770,12390,4654,1341,1375,467869.09375,10204,8 nanometer
...,...,...,...,...,...,...,...,...,...,...
12,navis.TreeNeuron,EPG(PB08)_R8,5813040233,11766,3817,1248,1283,431016.09375,10,8 nanometer
13,navis.TreeNeuron,EPG(PB08)_R8,5813061251,9887,3226,999,1023,361935.28125,11,8 nanometer


In [42]:
sklist_pruned = navis.in_volume(eb_sklist, EB, inplace=False)

Subsetting:   0%|          | 0/14 [00:00<?, ?it/s]

In [ ]:
# verify this worked
navis.plot3d(
    [sklist_pruned[0:3], EB],
    connectors=True
)

In [45]:
all_pts = np.vstack([
    n.nodes[['x', 'y', 'z']].to_numpy(dtype=float)
    for n in sklist_pruned
])

# in case duplicates
# all_pts = np.unique(all_pts, axis=0)

hull = ConvexHull(all_pts)

EB_2col = trimesh.Trimesh(vertices=all_pts, faces=hull.simplices, process=False)

navis.plot3d([EB, EB_2col])

In [49]:
# download all HD Neurons and prune to new subvolume

bodyid_list = fly_syn_df['bodyId'].tolist()

nlist = neu.fetch_skeletons( # download full skeletons
    neu.SegmentCriteria(bodyId=bodyid_list, regex=True), with_synapses=True
)

nlist

Fetching:   0%|          | 0/422 [00:00<?, ?it/s]

,type,name,id,n_nodes,n_connectors,n_branches,n_leafs,cable_length,soma,units
0,navis.TreeNeuron,PEN_b(PB06b)_L4,387023620,8111,2056,938,961,283142.81250,7838.0,8 nanometer
1,navis.TreeNeuron,PEN_a(PB06a)_L7,508793049,5295,2194,845,870,254623.00000,6.0,8 nanometer
...,...,...,...,...,...,...,...,...,...,...
420,navis.TreeNeuron,PEN_a(PB06a)_L5,5813080979,7463,2199,1071,1097,310830.68750,9.0,8 nanometer
421,navis.TreeNeuron,EPG(PB08)_R3,5813080838,12382,4750,1356,1392,460598.90625,11.0,8 nanometer


In [ ]:
nlist

In [53]:
nlist[0].connectors

,connector_id,node_id,type,x,y,z,roi,confidence
0,0,6778,post,23757,23753,20410,EB,0.614822
1,1,5724,pre,23000,24756,21601,EB,0.985000
2,2,5445,pre,25196,18879,22069,NaN,0.897000
3,3,5620,pre,22613,24710,21010,EB,0.998000
4,4,1324,post,30158,15773,13710,PB,0.985802
...,...,...,...,...,...,...,...,...
2051,2051,5512,post,22741,24806,21330,EB,0.930455
2052,2052,5910,post,22359,24716,20884,EB,0.971615
2053,2053,5611,post,22675,25334,21279,EB,0.919471
2054,2054,5765,post,23119,25084,20643,EB,0.942488


In [ ]:
fly_EB_filt = fly_syn_df[fly_syn_df['roi'].str.contains(r'^EB$')]
epg = r'EPGt_R9|EPG_R9|EPG_R8|EPG_L1|EPG_L2'
peg = r'PEG_R9|PEG_R8|PEG_L1|PEG_L2'
# pen_a = r'PEN_a_R9|PEN_a_R8|PEN_a_L2|PEN_a_L3'
# pen_b = r'PEN_b_R9|PEN_b_R8|PEN_b_L2|PEN_b_L3'

    
fly_EB_filt = fly_EB_filt[(fly_EB_filt['type_col'].str.contains(epg, case=False) | #filter type_pre_colaptic partner
                    fly_EB_filt['type_col'].str.contains(peg, case=False)
                    )]

In [ ]:
fly_EB_filt = fly_syn_df[fly_syn_df['roi'].str.contains(r'^EB$')]


In [ ]:
# need to filter to 

### Generate synapse table with pre, post, pre coord, post coord, pre name, post name

In [ ]:
# import synapse dataframe to get bodyIDs (can also just use neuron criteria and download)
# fly_syn_df = pd.read_csv('../syntables/table_csv_files/fly_hemibrain_all_hd_cells_syntable.csv')

In [ ]:
# # get EB synapses 

# fly_EB_filt = fly_syn_df[fly_syn_df['roi'].str.contains(r'^EB$')]
# epg = r'EPGt_R9|EPG_R9|EPG_R8|EPG_L1|EPG_L2'
# peg = r'PEG_R9|PEG_R8|PEG_L1|PEG_L2'
# pen_a = r'PEN_a_R9|PEN_a_R8|PEN_a_L2|PEN_a_L3'
# pen_b = r'PEN_b_R9|PEN_b_R8|PEN_b_L2|PEN_b_L3'

    
# fly_EB_filt = fly_EB_filt[(fly_EB_filt['type_col'].str.contains(epg, case=False) | #filter type_pre_colaptic partner
#                     fly_EB_filt['type_col'].str.contains(peg, case=False)
#                     )]

In [94]:
n = NC(type='EPG.*|PEG.*|PEN_a.*|PEN_b.*|Delta7.*|ER.*', regex = True)
n_info, n_conn = neu.fetch_neurons(n)

In [139]:
PB_col = n_info[n_info['instance'].str.contains(r'^(EPG|PEG|PEN).*_L[3-6]$')]
PB_d7 = n_info[n_info['instance'].str.contains(r'Delta')]

pbcol_list = PB_col['bodyId'].unique().tolist()
PB_d7_list = PB_d7['bodyId'].unique().tolist()

pat = r'^(EPG|EPGt|PEG).*_(L[1-2]|R[8-9])$|^PEN.*_(L[2-3]|R[8-9])$'

EB_col = n_info[n_info['instance'].str.contains(pat, regex=True)]
EB_er = n_info[n_info['instance'].str.contains(r'ER', regex=True)]

ebcol_list = EB_col['bodyId'].unique().tolist()
eber_list = EB_er['bodyId'].unique().tolist()

bodyId_list = pbcol_list + PB_d7_list + ebcol_list + eber_list

bodyId_list = list(set(bodyId_list))

bodyId_list

C:\Users\Marcel\AppData\Local\Temp\ipykernel_10532\564209064.py:1: UserWarning:

This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.

C:\Users\Marcel\AppData\Local\Temp\ipykernel_10532\564209064.py:9: UserWarning:

This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.



[1601771527,
 973959177,
 942522378,
 1106180107,
 1261086734,
 1199712272,
 1322133531,
 1322129437,
 1261086755,
 1261086756,
 1261764651,
 1169260589,
 1260828727,
 1198680122,
 1231079482,
 1230712894,
 1293135946,
 1292847181,
 5812983887,
 1322137688,
 1291444314,
 911911004,
 1261086823,
 911906936,
 1230712956,
 1261768832,
 1167632513,
 634962055,
 1380495496,
 1261086863,
 5813047446,
 1261768859,
 1261772956,
 1229965474,
 1261768880,
 1136861363,
 912601268,
 1292816565,
 1198680249,
 910796989,
 1293136062,
 1322129610,
 5813080269,
 1261773006,
 5813080272,
 1261768913,
 880875736,
 5812984024,
 910442723,
 819828986,
 5813059834,
 5813059835,
 910442752,
 1199427845,
 1168064774,
 1322133768,
 1168392468,
 911911193,
 910442782,
 1261773104,
 1261091124,
 1231675708,
 541870397,
 1322129724,
 1292458303,
 1322133825,
 1322129730,
 1260327246,
 880875861,
 1261091157,
 5813047647,
 1322133858,
 1261091174,
 1198680426,
 1167563121,
 1322129781,
 1198680441,
 1292458366,
 

In [140]:
rois = ['EB', 'PB', 'NO'] # 'PB(L3)', 'PB(L4)', 'PB(L5)', 'PB(L6)'

n_criteria = NC(bodyId=bodyId_list, regex=True, rois=rois, roi_req='any')

s_criteria = SC(rois=rois, primary_only=True) # include_nonprimary=True

syntable = fetch_synapse_connections(n_criteria, n_criteria, s_criteria)
syntable.head()

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/300230 [00:00<?, ?it/s]

,bodyId_pre,bodyId_post,roi_pre,roi_post,x_pre,y_pre,z_pre,x_post,y_post,z_post,confidence_pre,confidence_post
0,5813080979,387023620,NO,NO,25513,23688,23388,25517,23679,23397,0.929,0.880456
1,5813070465,387023620,NO,NO,24835,22927,23457,24834,22913,23444,0.778,0.813000
2,5813070465,387023620,NO,NO,25133,22742,23248,25127,22717,23240,0.912,0.870472
3,5813070465,387023620,NO,NO,24505,22960,22950,24525,22987,22964,0.966,0.643949
4,5813070465,387023620,NO,NO,24800,22873,23390,24821,22868,23385,0.883,0.663000


In [144]:
syns = neu.merge_neuron_properties(n_info, syntable, ['type', 'instance'])
syns

,bodyId_pre,bodyId_post,roi_pre,roi_post,x_pre,y_pre,z_pre,x_post,y_post,z_post,confidence_pre,confidence_post,type_pre,instance_pre,type_post,instance_post
0,5813080979,387023620,NO,NO,25513,23688,23388,25517,23679,23397,0.929,0.880456,PEN_a(PEN1),PEN_a(PB06a)_L5,PEN_b(PEN2),PEN_b(PB06b)_L4
1,5813070465,387023620,NO,NO,24835,22927,23457,24834,22913,23444,0.778,0.813000,PEN_b(PEN2),PEN_b(PB06b)_L3,PEN_b(PEN2),PEN_b(PB06b)_L4
2,5813070465,387023620,NO,NO,25133,22742,23248,25127,22717,23240,0.912,0.870472,PEN_b(PEN2),PEN_b(PB06b)_L3,PEN_b(PEN2),PEN_b(PB06b)_L4
3,5813070465,387023620,NO,NO,24505,22960,22950,24525,22987,22964,0.966,0.643949,PEN_b(PEN2),PEN_b(PB06b)_L3,PEN_b(PEN2),PEN_b(PB06b)_L4
4,5813070465,387023620,NO,NO,24800,22873,23390,24821,22868,23385,0.883,0.663000,PEN_b(PEN2),PEN_b(PB06b)_L3,PEN_b(PEN2),PEN_b(PB06b)_L4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
300225,387023620,5813080979,NO,NO,25036,22370,23166,25044,22370,23157,0.627,0.999000,PEN_b(PEN2),PEN_b(PB06b)_L4,PEN_a(PEN1),PEN_a(PB06a)_L5
300226,387023620,5813080979,NO,NO,25392,22741,23505,25428,22735,23517,0.905,0.594787,PEN_b(PEN2),PEN_b(PB06b)_L4,PEN_a(PEN1),PEN_a(PB06a)_L5
300227,387023620,5813080979,NO,NO,25752,23233,23146,25743,23238,23136,0.664,0.965000,PEN_b(PEN2),PEN_b(PB06b)_L4,PEN_a(PEN1),PEN_a(PB06a)_L5
300228,387023620,5813080979,NO,NO,24742,23104,23375,24733,23117,23364,0.864,0.781266,PEN_b(PEN2),PEN_b(PB06b)_L4,PEN_a(PEN1),PEN_a(PB06a)_L5


In [145]:
# format table
fly_syns = syns.copy()

# ####################################
# ####################################
# Regex patterns
pattern_standard = r'^([A-Z]{3}[a-zA-Z]?(?:\d+)?(?:_[a-z])?).*(R\d|L\d)'
pattern_er = r'^(ER\d+[a-z]?)(?:_([a-z]))?\(ring\)_([LR])$'
pattern_d7 = r'(?i)(Delta7).*?_(L\d(?:[LR]\d)+)'
pattern_lno = r'(LNO3|LNO2|LNO1|LNOa|GLNO)'

# Extract for standard neurons
extracted_pre_standard = fly_syns['instance_pre'].str.extract(pattern_standard)
extracted_post_standard = fly_syns['instance_post'].str.extract(pattern_standard)

# Extract for delta7 neurons
extracted_pre_d7 = fly_syns['instance_pre'].str.extract(pattern_d7)
extracted_post_d7 = fly_syns['instance_post'].str.extract(pattern_d7)

# Extract for lno neurons
extracted_lno_pre = fly_syns['type_pre'].str.extract(pattern_lno)
extracted_lno_post = fly_syns['type_post'].str.extract(pattern_lno)

# Extract for ER neurons
extracted_pre_er = fly_syns['instance_pre'].str.extract(pattern_er)
extracted_post_er = fly_syns['instance_post'].str.extract(pattern_er)


# precompute the fully constructed strings for each pattern
d7_type_pre = extracted_pre_d7[0] + '_' + extracted_pre_d7[1]
d7_type_post = extracted_post_d7[0] + '_' + extracted_post_d7[1]

er_type_pre = extracted_pre_er[0] + extracted_pre_er[1].fillna('') + '_' + extracted_pre_er[2]
er_type_post = extracted_post_er[0] + extracted_post_er[1].fillna('') + '_' + extracted_post_er[2]

std_type_pre = extracted_pre_standard[0] + '_' + extracted_pre_standard[1]
std_type_post = extracted_post_standard[0] + '_' + extracted_post_standard[1]


# assign type (e.g., only the class, not location info)
fly_syns['type_pre'] = extracted_pre_er[0].combine_first(extracted_pre_standard[0]).combine_first(extracted_pre_d7[0]).combine_first(extracted_lno_pre[0])
fly_syns['type_post'] = extracted_post_er[0].combine_first(extracted_post_standard[0]).combine_first(extracted_post_d7[0]).combine_first(extracted_lno_post[0])

fly_syns['type_pre_col'] = er_type_pre.combine_first(std_type_pre).combine_first(d7_type_pre).combine_first(extracted_lno_pre[0])
fly_syns['type_post_col'] = er_type_post.combine_first(std_type_post).combine_first(d7_type_post).combine_first(extracted_lno_post[0])

# # final naming
fly_syns['pre_name'] = fly_syns['type_pre_col'] + '_' + fly_syns['bodyId_pre'].astype(str)
fly_syns['post_name'] = fly_syns['type_post_col'] + '_' + fly_syns['bodyId_post'].astype(str)

fly_syns

,bodyId_pre,bodyId_post,roi_pre,roi_post,x_pre,y_pre,z_pre,x_post,y_post,z_post,confidence_pre,confidence_post,type_pre,instance_pre,type_post,instance_post,type_pre_col,type_post_col,pre_name,post_name
0,5813080979,387023620,NO,NO,25513,23688,23388,25517,23679,23397,0.929,0.880456,PEN_a,PEN_a(PB06a)_L5,PEN_b,PEN_b(PB06b)_L4,PEN_a_L5,PEN_b_L4,PEN_a_L5_5813080979,PEN_b_L4_387023620
1,5813070465,387023620,NO,NO,24835,22927,23457,24834,22913,23444,0.778,0.813000,PEN_b,PEN_b(PB06b)_L3,PEN_b,PEN_b(PB06b)_L4,PEN_b_L3,PEN_b_L4,PEN_b_L3_5813070465,PEN_b_L4_387023620
2,5813070465,387023620,NO,NO,25133,22742,23248,25127,22717,23240,0.912,0.870472,PEN_b,PEN_b(PB06b)_L3,PEN_b,PEN_b(PB06b)_L4,PEN_b_L3,PEN_b_L4,PEN_b_L3_5813070465,PEN_b_L4_387023620
3,5813070465,387023620,NO,NO,24505,22960,22950,24525,22987,22964,0.966,0.643949,PEN_b,PEN_b(PB06b)_L3,PEN_b,PEN_b(PB06b)_L4,PEN_b_L3,PEN_b_L4,PEN_b_L3_5813070465,PEN_b_L4_387023620
4,5813070465,387023620,NO,NO,24800,22873,23390,24821,22868,23385,0.883,0.663000,PEN_b,PEN_b(PB06b)_L3,PEN_b,PEN_b(PB06b)_L4,PEN_b_L3,PEN_b_L4,PEN_b_L3_5813070465,PEN_b_L4_387023620
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
300225,387023620,5813080979,NO,NO,25036,22370,23166,25044,22370,23157,0.627,0.999000,PEN_b,PEN_b(PB06b)_L4,PEN_a,PEN_a(PB06a)_L5,PEN_b_L4,PEN_a_L5,PEN_b_L4_387023620,PEN_a_L5_5813080979
300226,387023620,5813080979,NO,NO,25392,22741,23505,25428,22735,23517,0.905,0.594787,PEN_b,PEN_b(PB06b)_L4,PEN_a,PEN_a(PB06a)_L5,PEN_b_L4,PEN_a_L5,PEN_b_L4_387023620,PEN_a_L5_5813080979
300227,387023620,5813080979,NO,NO,25752,23233,23146,25743,23238,23136,0.664,0.965000,PEN_b,PEN_b(PB06b)_L4,PEN_a,PEN_a(PB06a)_L5,PEN_b_L4,PEN_a_L5,PEN_b_L4_387023620,PEN_a_L5_5813080979
300228,387023620,5813080979,NO,NO,24742,23104,23375,24733,23117,23364,0.864,0.781266,PEN_b,PEN_b(PB06b)_L4,PEN_a,PEN_a(PB06a)_L5,PEN_b_L4,PEN_a_L5,PEN_b_L4_387023620,PEN_a_L5_5813080979


In [146]:
# make back up
fly_syns.to_csv('../syntables/table_csv_files/fly_hemibrain_beeROI_syntable.csv')

### make subvolumes of EB and PB and filter synapses within
- NOTE: this isn't entirely necessary for PB...hemibrain already has PB annotations for PB columns. 

In [ ]:
# temp GALL counts
# gall = pd.read_csv('../syntables/table_csv_files/fly_hemibrain_all_hd_cells_syntable_simplified.csv')
# gall[gall['roi'].str.contains(r'GA')]

In [5]:
n = NC(type='EPG.*|PEG.*|PEN_a.*|PEN_b.*|Delta7.*|ER.*', regex = True)
n_info, n_conn = neu.fetch_neurons(n)

EB = n_neu.fetch_roi("EB")
PB = n_neu.fetch_roi("PB")

In [42]:
fly_syns = pd.read_csv('../syntables/table_csv_files/fly_hemibrain_all_hd_cells_syntable.csv')

In [44]:
# filter for neurons that will define the EB subvolume (EPG)
# obtain bodyIds for these
epg = r'^(EPG|EPGt|PEG).*_(L[1-2]|R[8-9])$' # r'EPGt_R9|EPG_R9|EPG_R8|EPG_L1|EPG_L2|PEG_R9|PEG_R8|PEG_L1|PEG_L2'

EB_col = n_info[n_info['instance'].str.contains(epg, regex=True)]

ebcol_list = EB_col['bodyId'].unique().tolist()

ebcol_list = list(set(ebcol_list))

ebcol_list

C:\Users\Marcel\AppData\Local\Temp\ipykernel_22764\1224635646.py:5: UserWarning:

This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.



[5813061251,
 1126647624,
 5813040233,
 697001770,
 5813059834,
 572870540,
 1190171884,
 1125964814,
 912545106,
 1128088594,
 1447576662,
 1125969082,
 1168659995,
 1219069439]

In [45]:
# repeat for PB
PB_col = n_info[n_info['instance'].str.contains(r'^(EPG|PEG|PEN).*_L[3-6]$')]

pbcol_list = PB_col['bodyId'].unique().tolist()

pbcol_list

C:\Users\Marcel\AppData\Local\Temp\ipykernel_22764\1914329357.py:2: UserWarning:

This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.



[387023620,
 387364605,
 449438847,
 539462336,
 541870397,
 634962055,
 664645558,
 664650297,
 665314820,
 696362840,
 757055317,
 758419409,
 788794171,
 789126240,
 819828986,
 849482511,
 879183317,
 880875861,
 881898734,
 912601268,
 942587380,
 974300015,
 1380495496,
 1508334312,
 1631450739,
 5813022281,
 5813070465,
 5813080979]

In [ ]:
# I want to use neuron branches to define subvolumes, so I need bodyIds to download from neuprint

In [46]:
eb_sklist = n_neu.fetch_skeletons( # download full skeletons
    n_neu.SegmentCriteria(bodyId=ebcol_list, regex=True), with_synapses=True
)
pb_sklist = n_neu.fetch_skeletons( # download full skeletons
    n_neu.SegmentCriteria(bodyId=pbcol_list, regex=True), with_synapses=True
)

eb_sklist

Fetching:   0%|          | 0/14 [00:00<?, ?it/s]

Fetching:   0%|          | 0/28 [00:00<?, ?it/s]

,type,name,id,n_nodes,n_connectors,n_branches,n_leafs,cable_length,soma,units
0,navis.TreeNeuron,EPG(PB08)_L1,572870540,14361,5572,1616,1658,541049.875000,14150,8 nanometer
1,navis.TreeNeuron,EPG(PB08)_R8,1125964814,10731,3398,1030,1056,371093.781250,9686,8 nanometer
...,...,...,...,...,...,...,...,...,...,...
12,navis.TreeNeuron,EPG(PB08)_R8,5813061251,9887,3226,999,1023,361935.281250,11,8 nanometer
13,navis.TreeNeuron,EPGt(PB09)_R9,1168659995,3071,765,377,390,141506.171875,2984,8 nanometer


In [47]:
eb_pruned = navis.in_volume(eb_sklist, EB, inplace=False)
pb_pruned = navis.in_volume(pb_sklist, PB, inplace=False)

Subsetting:   0%|          | 0/14 [00:00<?, ?it/s]

Copy:   0%|          | 0/28 [00:00<?, ?it/s]

Subsetting:   0%|          | 0/28 [00:00<?, ?it/s]

In [48]:
# drop (this has a single branch that extends outside of EBc1/2)
drop_idx = {1, 11}

# id to drop
drop_id = 5813040233

eb_pruned = navis.NeuronList(
    n for i, n in enumerate(eb_pruned)
    if i not in drop_idx and n.id != drop_id
)


In [ ]:
# verify this worked
navis.plot3d(
    [eb_pruned, EB],
    connectors=False
)

In [ ]:
# verify this worked
navis.plot3d(
    [pb_pruned, PB],
    connectors=True
)

In [49]:
# create mesh subvolumes
all_eb_pts = np.vstack([
    n.nodes[['x', 'y', 'z']].to_numpy(dtype=float)
    for n in eb_pruned
])

# in case duplicates
# all_pts = np.unique(all_pts, axis=0)
eb_hull = ConvexHull(all_eb_pts)
EB_2col = trimesh.Trimesh(vertices=all_eb_pts, faces=eb_hull.simplices, process=False)

########################
all_pb_pts = np.vstack([
    n.nodes[['x', 'y', 'z']].to_numpy(dtype=float)
    for n in pb_pruned
])

# in case duplicates
# all_pts = np.unique(all_pts, axis=0)
pb_hull = ConvexHull(all_pb_pts)
PB_4col = trimesh.Trimesh(vertices=all_pb_pts, faces=pb_hull.simplices, process=False)

In [ ]:
# figure image
navis.plot3d([EB, EB_2col, PB, PB_4col, eb_sklist, pb_sklist])

In [50]:
# takes a minute
EBq = EB_2col.copy()
EBq.process(validate=True)

PBq = PB_4col.copy()
PBq.process(validate=True)

# Build (N,3) arrays of synapse terminal coords
pre_pts  = fly_syns[['x_pre',  'y_pre',  'z_pre']].to_numpy(dtype=float)
post_pts = fly_syns[['x_post', 'y_post', 'z_post']].to_numpy(dtype=float)

# In either mesh (union)
pre_in  = EBq.contains(pre_pts)  | PBq.contains(pre_pts)
post_in = EBq.contains(post_pts) | PBq.contains(post_pts)

# Keep only rows where BOTH terminals are inside the union
fly_syns_in = fly_syns[pre_in & post_in].copy()

In [54]:
epgcheck = fly_syns_in[fly_syns_in['pre_name'].str.contains(r'PEG')]
epgcheck['pre_name'].unique()

array(['PEG_R8_5813059834', 'PEG_L2_1190171884', 'PEG_L3_974300015',
       'PEG_L4_942587380', 'PEG_L6_881898734', 'PEG_L1_1128088594',
       'PEG_R9_1125969082'], dtype=object)

In [61]:
# subvolumes aren't perfect, so remove any neurons that aren't included in bee but maybe innervate the subvolume partially
fly_filt = fly_syns_in.copy()
pat = r'EPG_R1|PEG_R1|EPG_L3|PEG_L3|EPG_L4|PEG_L4|EPG_R7|PEG_R7|PEN_a_L4|PEN_a_L5|PEN_b_L4|PEN_a_R7|PEN_b_R7'

eb_filt = (
    ((fly_filt['roi_pre'] == 'EB') &
     (fly_filt['type_pre_col'].str.contains(pat, regex=True, na=False))) |
    ((fly_filt['roi_post'] == 'EB') &
     (fly_filt['type_post_col'].str.contains(pat, regex=True, na=False)))
)

fly_filt = fly_filt[~eb_filt].copy()


pat = r'EPG_L2|PEG_L2|EPG_L7|PEG_L7|PEN_a_L2|PEN_b_L2|PEN_a_L7|PEN_b_L7'

pb_filt = (
    ((fly_filt['roi_pre'] == 'PB') &
     (fly_filt['type_pre_col'].str.contains(pat, regex=True, na=False))) |
    ((fly_filt['roi_post'] == 'PB') &
     (fly_filt['type_post_col'].str.contains(pat, regex=True, na=False)))
)

fly_filt = fly_filt[~pb_filt].copy()

In [57]:
epgl3 = fly_filt[fly_filt['bodyId_pre'].isin([387364605])]
peg5 = fly_syns[fly_syns['bodyId_pre'].isin([880875861])]
epgl3 = epgl3[['x_pre', 'y_pre', 'z_pre']].rename(columns={'x_pre': 'x', 'y_pre': 'y', 'z_pre': 'z'})
peg5 = peg5[['x_pre', 'y_pre', 'z_pre']].rename(columns={'x_pre': 'x', 'y_pre': 'y', 'z_pre': 'z'})


In [ ]:
navis.plot3d([EB, EB_2col, PB, PB_4col, peg5])

In [ ]:
navis.plot3d([EB, EB_2col, PB, PB_4col, epgl3])

In [62]:
fly_filt.to_csv('../syntables/table_csv_files/fly_hemibrain_beeROI_syntable.csv')

### make connectivity table from fly bee ROI filtered table


In [5]:
fly_count = pd.read_csv('../syntables/table_csv_files/fly_hemibrain_beeROI_syntable.csv')
fly_count

,Unnamed: 0.1,Unnamed: 0,bodyId_pre,bodyId_post,roi_pre,roi_post,x_pre,y_pre,z_pre,x_post,...,confidence_pre,confidence_post,type_pre,instance_pre,type_post,instance_post,type_pre_col,type_post_col,pre_name,post_name
0,12,12,5813048042,387023620,PB,PB,30662,16090,13159,30636,...,0.969,0.454083,Delta7,Delta7(PB15)_L5R4_L,PEN_b,PEN_b(PB06b)_L4,Delta7_L5R4,PEN_b_L4,Delta7_L5R4_5813048042,PEN_b_L4_387023620
1,35,35,5813022281,387023620,PB,PB,30200,15476,13766,30200,...,0.918,0.912754,EPG,EPG(PB08)_L4,PEN_b,PEN_b(PB06b)_L4,EPG_L4,PEN_b_L4,EPG_L4_5813022281,PEN_b_L4_387023620
2,36,36,5813022281,387023620,PB,PB,28788,14632,14350,28782,...,0.943,0.916443,EPG,EPG(PB08)_L4,PEN_b,PEN_b(PB06b)_L4,EPG_L4,PEN_b_L4,EPG_L4_5813022281,PEN_b_L4_387023620
3,37,37,5813022281,387023620,PB,PB,30066,16005,13434,30078,...,0.895,0.979737,EPG,EPG(PB08)_L4,PEN_b,PEN_b(PB06b)_L4,EPG_L4,PEN_b_L4,EPG_L4_5813022281,PEN_b_L4_387023620
4,38,38,5813022281,387023620,PB,PB,29844,15630,13850,29831,...,0.947,0.416655,EPG,EPG(PB08)_L4,PEN_b,PEN_b(PB06b)_L4,EPG_L4,PEN_b_L4,EPG_L4_5813022281,PEN_b_L4_387023620
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100655,300123,300123,696362840,5813080979,PB,PB,31573,14792,13752,31575,...,0.977,0.710217,EPG,EPG(PB08)_L5,PEN_a,PEN_a(PB06a)_L5,EPG_L5,PEN_a_L5,EPG_L5_696362840,PEN_a_L5_5813080979
100656,300124,300124,696362840,5813080979,PB,PB,31661,15488,13082,31664,...,0.950,0.462977,EPG,EPG(PB08)_L5,PEN_a,PEN_a(PB06a)_L5,EPG_L5,PEN_a_L5,EPG_L5_696362840,PEN_a_L5_5813080979
100657,300125,300125,696362840,5813080979,PB,PB,30601,15904,13339,30578,...,0.898,0.412089,EPG,EPG(PB08)_L5,PEN_a,PEN_a(PB06a)_L5,EPG_L5,PEN_a_L5,EPG_L5_696362840,PEN_a_L5_5813080979
100658,300126,300126,696362840,5813080979,PB,PB,31326,15114,14185,31318,...,0.723,0.304000,EPG,EPG(PB08)_L5,PEN_a,PEN_a(PB06a)_L5,EPG_L5,PEN_a_L5,EPG_L5_696362840,PEN_a_L5_5813080979


In [9]:
fly_count = fly_count[fly_count['bodyId_pre'] != fly_count['bodyId_post']] #remove autapses

df = fly_count[['bodyId_pre', 'bodyId_post', 'pre_name', 'post_name', 'type_pre_col', 'type_post_col', 'type_pre', 'type_post', 'roi_pre', 'roi_post']]

'''keep NaN partners so that relative weight score is calculated with total synapse numbers 
makes it comparable to the fruit fly value_counts() here counts how many times each unique 
combination of the 4 columns occurs in the df'''
fly_conn = df.value_counts(dropna=False)

# Reset index turns MultiIndex into columns, and .rename() makes column names explicit
fly_conn = fly_conn.reset_index().rename(
    columns={
        0: 'count'  # This is the name of the column created by value_counts()
    }
)

fly_conn

,bodyId_pre,bodyId_post,pre_name,post_name,type_pre_col,type_post_col,type_pre,type_post,roi_pre,roi_post,count
0,910783961,5813070465,Delta7_L3R6_910783961,PEN_b_L3_5813070465,Delta7_L3R6,PEN_b_L3,Delta7,PEN_b,PB,PB,180
1,911565419,5813070465,Delta7_L3R6_911565419,PEN_b_L3_5813070465,Delta7_L3R6,PEN_b_L3,Delta7,PEN_b,PB,PB,141
2,634608104,572870540,PEN_a_L2_634608104,EPG_L1_572870540,PEN_a_L2,EPG_L1,PEN_a,EPG,EB,EB,126
3,880875736,1631450739,Delta7_L5R4_880875736,PEN_b_L5_1631450739,Delta7_L5R4,PEN_b_L5,Delta7,PEN_b,PB,PB,125
4,665314820,942587380,EPG_L4_665314820,PEG_L4_942587380,EPG_L4,PEG_L4,EPG,PEG,PB,PB,119
...,...,...,...,...,...,...,...,...,...,...,...
16234,1261086863,1261432168,ER3da_L_1261086863,ER3dd_R_1261432168,ER3da_L,ER3dd_R,ER3d,ER3d,EB,EB,1
16235,1261086863,1261432159,ER3da_L_1261086863,ER3db_L_1261432159,ER3da_L,ER3db_L,ER3d,ER3d,EB,EB,1
16236,1261086863,1261432076,ER3da_L_1261086863,ER3db_L_1261432076,ER3da_L,ER3db_L,ER3d,ER3d,EB,EB,1
16237,1261086863,1261423507,ER3da_L_1261086863,ER3dc_R_1261423507,ER3da_L,ER3dc_R,ER3d,ER3d,EB,EB,1


In [10]:
fly_conn.to_csv('../syntables/table_csv_files/fly_hemibrain_beeROI_conntable.csv', index=False)

### obtain ER counts for fly (for Figure 1 stacked bar chart)

In [4]:
fly_count = pd.read_csv("./table_csv_files/fly_hemibrain_all_hd_cells_conntable.csv")

In [8]:
# this function assigns a BU sub-roi to ER neurons 
# based on Hulse et al. 2021 Table 5

def normalize_er_types(df, pre_col='type_pre', post_col='type_post'):
    pattern_map = {
        r'^ER1.*': 'ER1_3a_LAL',
        r'^ER2.*': 'ER_BUs',
        r'^ER3w.*': 'ER_BUs',
        r'^ER4d.*': 'ER_BUs',
        r'^ER3a.*': 'ER1_3a_LAL',
        r'^ER3d.*': 'ER_BUi',
        r'^ER3p.*': 'ER_BUi',
        r'^ER3m.*': 'ER_BUi',
        r'^ER4m.*': 'ER_BUa',
        r'^ER5.*':  'ER_BUs',
        r'^ER6.*': "ER_BUa",
        r'^ExR.*':  'ExR'
    }
    
    # Start by copying original columns
    df = df.copy()
    df['original_type_pre'] = df[pre_col]
    df['original_type_post'] = df[post_col]

    # Apply replacements only to new columns
    for pattern, replacement in pattern_map.items():
        df[pre_col] = df[pre_col].str.replace(pattern, replacement, regex=True)
        df[post_col] = df[post_col].str.replace(pattern, replacement, regex=True)


    return df

In [9]:
# obtain ER counts for fly (for Figure 1 stacked bar chart)
fly_er = normalize_er_types(fly_count)


# 1. Keep only rows where BOTH partners are ER neurons
er_df = fly_er[
    fly_er["pre_name"].str.contains("ER", na=False) &
    fly_er["post_name"].str.contains("ER", na=False)
].copy()

# 2. Build a unique neuron table from pre and post columns
pre_neurons = er_df[["pre_name", "type_pre"]].copy()
pre_neurons.columns = ["neuron", "roi"]

post_neurons = er_df[["post_name", "type_post"]].copy()
post_neurons.columns = ["neuron", "roi"]

all_neurons = pd.concat([pre_neurons, post_neurons], ignore_index=True)

# Drop exact duplicates
all_neurons = all_neurons.drop_duplicates()

# If the same neuron appears multiple times with the same ROI, this is enough.
# If a neuron somehow appears with conflicting ROI labels, this will keep both rows.
# You can inspect those with:
conflicts = (
    all_neurons.groupby("neuron")["roi"]
    .nunique()
    .reset_index(name="n_roi")
)
conflicts = conflicts[conflicts["n_roi"] > 1]

# 3. Assign side from neuron name
all_neurons["side"] = np.where(
    all_neurons["neuron"].str.contains("_R", na=False), "right",
    np.where(all_neurons["neuron"].str.contains("_L", na=False), "left", np.nan)
)

# Optional: drop neurons where side could not be determined
all_neurons = all_neurons.dropna(subset=["side"])

# 4. Count unique neurons by side and ROI
cell_counts = (
    all_neurons.groupby(["side", "roi"], as_index=False)
    .size()
    .rename(columns={"size": "cell_count"})
)

print(cell_counts)

    side         roi  cell_count
0   left  ER1_3a_LAL          29
1   left      ER_BUa           6
2   left      ER_BUi          39
3   left      ER_BUs          53
4  right  ER1_3a_LAL          27
5  right      ER_BUa           7
6  right      ER_BUi          41
7  right      ER_BUs          54


In [14]:
cell_counts[cell_counts["side"].str.contains(r"right")] # most complete

,side,roi,cell_count
4,right,ER1_3a_LAL,27
5,right,ER_BUa,7
6,right,ER_BUi,41
7,right,ER_BUs,54


In [12]:
cg = cell_counts.groupby("side", as_index=False)["cell_count"].sum()

cg

'''There are 129 right side ER in hemibrain and 139 right side ER cells in FAFB (not shown here)'''

,side,cell_count
0,left,127
1,right,129


In [ ]:
''' to obtain TuBu counts I used following Neo4j Cypher query on the neuprint.janelia.org 
hemibrain:v1.2.1:

total count
MATCH (n:Neuron)
WHERE n.type CONTAINS "TuBu"
  AND n.instance CONTAINS "_R"
RETURN count(n)

74

-------------------
TUBU map:
"BUs"

MATCH (n:Neuron)
WHERE any(x IN ["TuBu06","TuBu07","TuBu08","TuBu09","TuBu10"]
          WHERE n.instance CONTAINS x)
  AND n.instance ENDS WITH "_R"
RETURN count(n)

41

"BUi"
MATCH (n:Neuron)
WHERE any(x IN ["TuBu02","TuBu03","TuBu04","TuBu05"]
          WHERE n.instance CONTAINS x)
  AND n.instance ENDS WITH "_R"
RETURN count(n)

28

"BUa"
MATCH (n:Neuron)
WHERE any(x IN ["TuBu01"]
          WHERE n.instance CONTAINS x)
  AND n.instance ENDS WITH "_R"
RETURN count(n)

5
'''